In [19]:
# Extract_Cemeteries_Smith_Counts.py
# 7/4/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math# Extract_Cemeteries_smith_Counts.py
# 6/28/2025

from bs4 import BeautifulSoup
import datetime
from datetime import datetime as dt
from datetime import date
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
import json

import sys
import random

import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date):
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out
def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
       
def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S")
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index



def random_sleep_new(get_counter):
    global sleep_time_pause,sleep_range,pause_count
    get_counter+=1
    if get_counter%pause_count==0:
        sleep_time =sleep_time_pause
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds   Sleeping till: {sleep_end_date}")
    else:
        sleep_time = random.uniform(sleep_range[0],sleep_range[1])
    time.sleep(sleep_time)    
    return get_counter



def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header)
    status_code=response.status_code
    url_source = response.text
   
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser')
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text)
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    for iTry in range(0,2):
        response = requests.request("GET", url, headers=my_header)
        status_code=response.status_code
        url_source = response.text
        if status_code!=200:
            print('\tstatus_code:',status_code, 'url',url)
            time.sleep(60)
            continue
        beautifulSoupText = BeautifulSoup(url_source, 'html.parser')
        specific_tag_text = beautifulSoupText.get_text()
        capture_json=json.loads(specific_tag_text)
        try:
            grave_count=capture_json['total']
        except:
            specific_tag_text
            print(capture_json)
            print(f" XXXX get_grave_count Failed: Try: {iTry} \n url: {url}")
            time.sleep(60)
    return capture_json,grave_count
   
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic1
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}
# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++
blank_line='                                                                                                                                                                                           '

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Search smiths All Cemeteries '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
# +++++++++++++++++++++++++++++++ Header ++++++++++++++++++++++++++++++++++++++++

# +++++++++++++++++++ File Input and Output +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileInput="FAG_Cemetery_Levy_List"
fileOutput="FAG_Cemetery_Smith_Counts"

if append==1:fileIn='FAG_Cemetery_Full_List_X'

fileIn=DirMain+"OutputData\\Cemeteries\\"+fileInput+".csv"
fileOut=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"OutputData\\Cemeteries\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_Interim.csv"
# +++++++++++++++++++ File Input and Output +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# *************************** Read Input File and Process File +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
cemetery_df=pd.read_csv(fileIn, encoding='ISO-8859-1')


field_list=['cemetery_id', 'memorial_count', 'cemetery_name', 'cemetery_city', 'cemetery_county', 'cemetery_state', 'county_id', 'state_id']
field_list_set=set(field_list)
# *************************** Read Input File and Process File +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

url_skip_county = "https://www.findagrave.com/memorial/search?ajax=true&skip="
url_county_id ="&limit=100&firstname=&middlename=&lastname=smith&birthyear=&locationId="
url_suffix="&location=&exactName=true"

url_cemetery = "https://www.findagrave.com/cemetery/"
url_skip ="/memorial-search?ajax=true&skip="
url_suffix="&limit=100&firstname=&middlename=&lastname=smith&birthyear=&locationId=&location=&exactName=true"


sleep_range=[5,12]
sleep_time_pause=1*60
pause_count=100
get_counter=0


cemetery_smith_count_df=pd.DataFrame([])
cemetery_count=0
total_cemetery_count=len(cemetery_df)
cum_smith_count=0
print('\n')
for index, row in cemetery_df.iterrows():
    cemetery_count+=1
    cemetery_id=row['cemetery_id']
    cemetery_name=row['cemetery_name']
    cemetery_grave_count=row['memorial_count']
    cemetery_state=row['cemetery_state']
    cemetery_county=row['cemetery_county']
    
    url=url_cemetery+str(cemetery_id)+url_skip+'0'+url_suffix
    capture_json,cemetery_smith_count=get_grave_count(url)
    cum_smith_count+=cemetery_smith_count
    if cemetery_grave_count==0:
        cemetery_smith_percent=0
    else:
        cemetery_smith_percent=100*cemetery_smith_count/cemetery_grave_count
    # print(f" {blank_line}", end='\r')
    print(f"       ..... {cemetery_state} •• {cemetery_county} • {cemetery_name} • Cemetery Count: {cemetery_count}/{total_cemetery_count} "+
              f"• smith Count: {cum_smith_count} • {cemetery_smith_count}/{cemetery_grave_count}/{cemetery_smith_percent:.1f}% •• ")
   
    smith_count_df=pd.DataFrame([cemetery_state,cemetery_county,cemetery_name,cemetery_id,cemetery_grave_count,cemetery_smith_count]).T
    smith_count_df.columns=['state','county','cemetery_name','cemetery_id','cemetery_grave_count','smith_count']
    cemetery_smith_count_df = pd.concat([cemetery_smith_count_df,smith_count_df])
    cemetery_smith_count_df.to_csv(fileOutInt,mode="w", index=False,header=True)

    # print(cemetery_smith_count_df)
    # input('')
    get_counter=random_sleep_new(get_counter)
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")  

cemetery_smith_count_df.to_csv(fileOut,mode="w", index=False,header=True)
cemetery_smith_count_df.to_csv(fileOut0,mode="w", index=False,header=True)
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Search smiths All Cemeteries   *** , Time: 2025-07-06 07:10:58.611166  **



       ..... Alabama •• Dallas County • Live Oak Cemetery • Cemetery Count: 1/3313 • smith Count: 109 • 109/6774/1.6% •• 
       ..... Alabama •• Etowah County • Beth Israel Cemetery • Cemetery Count: 2/3313 • smith Count: 109 • 0/160/0.0% •• 
       ..... Alabama •• Etowah County • Crestwood Memorial Cemetery • Cemetery Count: 3/3313 • smith Count: 427 • 318/16010/2.0% •• 
       ..... Alabama •• Jefferson County • Elmwood Cemetery and Mausoleum • Cemetery Count: 4/3313 • smith Count: 2183 • 1756/125142/1.4% •• 
       ..... Alabama •• Jefferson County • Jefferson Memorial Gardens South • Cemetery Count: 5/3313 • smith Count: 2251 • 68/5265/1.3% •• 
       ..... Alabama •• Jefferson County • Knesses Israel Cemetery • Cemetery Count: 6/3313 • smith Count: 2251 • 0/1694/0.0% •• 
       ..... Alabama •• Jefferson County • Temple Emanu-El Cemetery • Cemetery Count: 7/3313 • smith Count: 2251 • 0/9

In [16]:
fileIn
FAG_Cemetery_Levy_List

'C:\\Users\\jrozm\\OneDrive\\Benjamin\\OutputData\\Cemeteries\\FAG_Cemetery_Levi_List.csv'